# Chapter 21: Measurements and Transformations

**Source span.** Perspectives on Projective Geometry, Chapter 21, Sections 21.1-21.6, printed pages 399-422 / PDF pages 421-444. The source was used for orientation, terminology, and coverage. The prose, diagrams, code, and checks below are original.

**Chapter goal.** Build a computational model for Cayley-Klein motions: measurements are read from an absolute conic, motions preserve that absolute, reference points on the absolute can be eliminated from the formulas, and reflections supply concrete generators whose products behave like rotations, translations, or hyperbolic analogues.

The chapter starts with a subtle point. A distance or angle measurement made from a cross-ratio is oriented if the two reference intersections with the absolute are ordered. When the line or pencil is not fixed in advance, that order is not intrinsic. In code this means that the usable projective datum is not one number `alpha`, but the reciprocal pair `{alpha, 1/alpha}`. Taking a logarithm turns this reciprocal choice into a sign choice; comparing cross-ratios directly avoids that extra branch.

The second theme is that a Cayley-Klein motion is best detected by the object it preserves. A projective matrix may distort a grid dramatically, but if it preserves the primal and dual fundamental conics, it also preserves the measurement formulas derived from them. The notebook uses this as the main invariant: diagrams show how points, lines, mirrors, and orbits move; the checks ask whether the conic form, cross-ratio class, and reference-free measurement value survived.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-21-measurements-and-transformations"
ARTIFACT_ROOT


## Computational Translation Guide

| Chapter object | Computational representation | What will be checked |
| --- | --- | --- |
| Fundamental object | symmetric matrices `A` and `B`; here the nondegenerate disk model uses `diag(1, 1, -1)` | `T.T @ A @ T` and `T @ B @ T.T` are scalar multiples of the original forms |
| Oriented measurement | cross-ratio `(p, q; X, Y)` on a chord or line pencil | reversing `X` and `Y` gives the reciprocal, not a new unoriented measurement |
| Reference-free distance comparison | `Omega_pq = p.T @ A @ q`, discriminant `Delta`, and `Psi = (p x q).T B (p x q)/(p.T A p q.T A q)` | direct endpoint computation matches the reciprocal class of the formula using only `p`, `q`, `A`, and `B` |
| K-reflection | projective involution with center `o`, mirror `m`, and pole/polar relation `m ~ A o`, `o ~ B m` | the matrix `<o,m>I - 2 o m.T` squares to a scalar identity and preserves the measurement |
| Rotation from reflections | product of two reflection matrices | finite mirror intersection gives an ordinary rotation; exterior intersection gives a hyperbolic-type iteration with fixed boundary directions |

**Storyboard implemented.** The notebook follows seven inspectable artifacts: oriented reciprocal measurement, absolute-preserving motion, proof dependency graph, reference-point elimination, measurement comparison level sets, pole/polar reflection, and a two-reflection rotation lab. Each artifact has a nearby invariant or residual in the final sanity JSON.


In [ ]:
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sympy as sp
from numpy.lib.scimath import sqrt as complex_sqrt

from utils.artifacts import (
    assert_artifacts,
    artifact_path,
    book_relative,
    display_artifact,
    ensure_artifact_root,
    image_stats,
    save_json,
    save_table,
)
from utils.projective import affine, cross_ratio, hpoint, join, meet

ensure_artifact_root(ARTIFACT_ROOT)
A = np.diag([1.0, 1.0, -1.0])
B = np.linalg.inv(A)
E3 = np.eye(3)
TOL = 1e-9

COLORS = {
    "ink": "#202124",
    "gray": "#73777f",
    "light": "#eef1f4",
    "blue": "#2f6fbb",
    "green": "#3f7f54",
    "red": "#b4463a",
    "gold": "#b98b18",
    "purple": "#6d4aa2",
}

artifact_paths = []
checks = {}


def rel(path):
    return book_relative(path)


def save_fig(fig, category, filename):
    path = artifact_path(ARTIFACT_ROOT, category, filename)
    fig.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    artifact_paths.append(path)
    return path


def omega(p, q, form=A):
    return float(np.asarray(p, dtype=float) @ form @ np.asarray(q, dtype=float))


def psi_value(p, q, primal=A, dual=B):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    line = np.cross(p, q)
    denom = float((p @ primal @ p) * (q @ primal @ q))
    return float((line @ dual @ line) / denom)


def line_conic_roots(p, q, form=A):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    direction = q - p
    aa = float(direction @ form @ direction)
    bb = float(2 * p @ form @ direction)
    cc = float(p @ form @ p)
    return np.roots([aa, bb, cc])


def chord_cross_ratio(p, q, form=A):
    roots = line_conic_roots(p, q, form)
    alpha = cross_ratio(0, 1, roots[0], roots[1])
    return alpha, roots


def reference_free_alpha(p, q, form=A):
    opp = omega(p, p, form)
    opq = omega(p, q, form)
    oqq = omega(q, q, form)
    delta = opq * opq - opp * oqq
    return (opq + complex_sqrt(delta)) / (opq - complex_sqrt(delta))


def reciprocal_class_error(alpha, beta):
    return float(min(abs(alpha - beta), abs(alpha * beta - 1.0)))


def plot_unit_circle(ax, **kwargs):
    theta = np.linspace(0, 2 * np.pi, 500)
    ax.plot(np.cos(theta), np.sin(theta), **kwargs)


def plot_projective_line(ax, line, xlim=(-1.4, 1.7), ylim=(-1.25, 1.25), **kwargs):
    a, b, c = [float(v) for v in line]
    if abs(b) >= abs(a):
        xs = np.array(xlim)
        ys = -(a * xs + c) / b
    else:
        ys = np.array(ylim)
        xs = -(b * ys + c) / a
    ax.plot(xs, ys, **kwargs)


def aff(point):
    point = np.asarray(point, dtype=float)
    return point[:2] / point[2]


def transform_points(matrix, xy_points):
    pts = np.column_stack([xy_points, np.ones(len(xy_points))])
    image = (matrix @ pts.T).T
    return image[:, :2] / image[:, 2:3]


def reflection_matrix(center, mirror):
    center = np.asarray(center, dtype=float)
    mirror = np.asarray(mirror, dtype=float)
    scale = float(mirror @ center)
    return scale * E3 - 2.0 * np.outer(center, mirror)


def reflection_from_mirror(mirror, dual=B):
    mirror = np.asarray(mirror, dtype=float)
    center = dual @ mirror
    return reflection_matrix(center, mirror), center


def mirror_line_through_angle(phi):
    return np.array([-math.sin(phi), math.cos(phi), 0.0])


def line_disk_segment(line):
    a, b, c = [float(v) for v in line]
    norm = math.hypot(a, b)
    foot = np.array([-a * c / norm**2, -b * c / norm**2])
    direction = np.array([-b, a]) / norm
    radius_sq = 1.0 - float(foot @ foot)
    if radius_sq <= 0:
        return np.empty((0, 2))
    span = math.sqrt(radius_sq)
    return np.vstack([foot - span * direction, foot + span * direction])


def det2(u, v):
    return float(u[0] * v[1] - u[1] * v[0])


def cross_ratio_homogeneous_2d(a, b, c, d):
    return det2(a, c) * det2(b, d) / (det2(a, d) * det2(b, c))


storyboard = {
    "chapter goal": "Explain which projective transformations preserve Cayley-Klein measurements and how reflections generate rotations.",
    "source span read": "Chapter 21, Sections 21.1-21.6; printed pp. 399-422; PDF pp. 421-444.",
    "concept inventory": [
        "measurements versus oriented measurements as reciprocal cross-ratio classes",
        "K-motions as projective transformations preserving the primal and dual fundamental forms",
        "elimination of the auxiliary absolute points X and Y through Omega and Delta",
        "comparison of measurements through the reference-free Psi expression",
        "projective involutions, mirror lines, centers, and harmonic conjugates",
        "pole/polar pairs for Cayley-Klein reflections",
        "products of two reflections as rotations, translations, glide reflections, or hyperbolic analogues",
    ],
    "library routing table": [
        {"concept": "2D projective conic diagrams", "representation": "static annotated PNG", "library": "Matplotlib", "why": "precise labels, equal aspect, and durable artifacts for incidence and pole/polar geometry"},
        {"concept": "symbolic reflection identities", "representation": "exact matrix checks", "library": "SymPy", "why": "reflection and rotation identities are small exact matrix equalities"},
        {"concept": "proof dependencies", "representation": "directed dependency graph", "library": "NetworkX", "why": "theorems depend on a short chain of preserved structures and invariants"},
        {"concept": "two-reflection lab", "representation": "interactive HTML", "library": "Plotly", "why": "orbit comparison and mirror toggling are easier to inspect interactively than in a fixed PNG"},
        {"concept": "numeric projective checks", "representation": "arrays and residuals", "library": "NumPy", "why": "homogeneous coordinates, cross products, and conic forms are linear-algebraic"},
    ],
    "visual sequence": [
        {"artifact": "figures/oriented-measurement-reciprocal.png", "inspection target": "reverse the conic endpoints and see alpha become 1/alpha", "validation": "reciprocal product residual"},
        {"artifact": "figures/absolute-preserving-motion.png", "inspection target": "watch a distorted grid preserve the absolute conic", "validation": "Lorentz conic residual and Psi residual"},
        {"artifact": "figures/motion-invariant-proof-dependencies.png", "inspection target": "trace which facts feed measurement invariance", "validation": "graph node and edge count"},
        {"artifact": "figures/reference-point-elimination.png", "inspection target": "compare endpoint cross-ratios with the Omega/Delta formula", "validation": "maximum reciprocal-class residual"},
        {"artifact": "figures/measurement-comparison-level-sets.png", "inspection target": "compare Euclidean and pseudo-Euclidean Psi level sets", "validation": "degenerate formula residuals"},
        {"artifact": "figures/pole-polar-reflection.png", "inspection target": "center, mirror, harmonic partner, and conic preservation", "validation": "involution, pole/polar, harmonic, and Psi residuals"},
        {"artifact": "html/two-reflections-rotation-lab.html", "inspection target": "compare finite-center rotation with exterior-center hyperbolic iteration", "validation": "rotation angle residual and orbit data"},
    ],
    "artifact plan": "All outputs live under artifacts/chapter-21-measurements-and-transformations/{figures,html,tables,checks}.",
    "computational checks": [
        "cross-ratio reciprocal ambiguity",
        "absolute-conic preservation by K-motions",
        "endpoint-free formula agrees up to reciprocal order",
        "Psi recovers Euclidean and pseudo-Euclidean squared distances in degenerate examples",
        "reflection matrix squares to scalar identity and preserves the conic form",
        "harmonic cross-ratio for a reflected point",
        "two line reflections rotate by twice the mirror angle",
        "artifact existence, nonzero size, and nonblank raster statistics",
    ],
    "proof-visualization strategy": "Use a NetworkX dependency graph plus symbolic reflection identities rather than reproducing the textbook proof text.",
    "risks": "Plotly HTML is generated locally; visual inspection in a browser gives the richest view, while nbclient validates the data and file output.",
}
storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
storyboard_path


## 1. Oriented Measurements Are Reciprocal Classes

A point distance in a Cayley-Klein geometry starts with a line through two points `p` and `q`. That line meets the absolute conic in two reference points `X` and `Y`. If a line was fixed in advance, an orientation could choose which one is first. For arbitrary point pairs there is no canonical order, so `(p, q; X, Y)` and `(p, q; Y, X)` must be treated as the same unoriented datum.

The diagram below makes the ambiguity visible. The chord and the two measured points do not change; only the names of the two conic intersections are swapped. The check beside the figure records that the two cross-ratios multiply to one.


In [ ]:
p = hpoint(-0.35, 0.10)
q = hpoint(0.55, 0.30)
alpha_xy, roots = chord_cross_ratio(p, q)
alpha_yx = cross_ratio(0, 1, roots[1], roots[0])
orientation_error = abs(alpha_xy * alpha_yx - 1.0)
X = p + roots[0] * (q - p)
Y = p + roots[1] * (q - p)
checks["orientation_reciprocal_error"] = float(orientation_error)
checks["oriented_alpha"] = float(np.real(alpha_xy))
checks["reversed_alpha"] = float(np.real(alpha_yx))

fig, ax = plt.subplots(figsize=(7.4, 5.2))
plot_unit_circle(ax, color=COLORS["blue"], linewidth=2.2, label="absolute conic")
plot_projective_line(ax, join(p, q), color=COLORS["gray"], linewidth=1.4, linestyle="--", label="join(p,q)")
for label, point, color in [("p", p, COLORS["ink"]), ("q", q, COLORS["ink"]), ("X", X, COLORS["red"]), ("Y", Y, COLORS["green"])]:
    xy = aff(point)
    ax.scatter([xy[0]], [xy[1]], s=70, color=color, zorder=5)
    ax.text(xy[0] + 0.035, xy[1] + 0.035, label, fontsize=10, weight="bold", color=color)
px, qx = aff(p), aff(q)
ax.plot([px[0], qx[0]], [px[1], qx[1]], color=COLORS["ink"], linewidth=3, alpha=0.8)
ax.annotate("oriented choice X -> Y", xy=aff(X), xytext=(-1.12, 0.78), arrowprops={"arrowstyle": "->", "color": COLORS["red"]}, color=COLORS["red"], fontsize=9)
ax.annotate("reversing endpoints gives reciprocal", xy=aff(Y), xytext=(-1.28, -0.78), arrowprops={"arrowstyle": "->", "color": COLORS["green"]}, color=COLORS["green"], fontsize=9)
ax.text(0.08, -1.20, f"alpha = {np.real(alpha_xy):.4f}   1/alpha = {np.real(alpha_yx):.4f}   product residual = {orientation_error:.1e}", fontsize=9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.35, 1.35)
ax.set_ylim(-1.32, 1.25)
ax.set_title("Unoriented measurement keeps the reciprocal cross-ratio class")
ax.legend(loc="upper right")
oriented_path = save_fig(fig, "figures", "oriented-measurement-reciprocal.png")
display_artifact(oriented_path, width=760)


## 2. Motions Preserve the Fundamental Object

For nondegenerate examples a motion can be tested by a matrix identity. In the disk model the absolute conic is `x^2 + y^2 - z^2 = 0`. A real matrix `T` with `T.T @ A @ T = A` maps the conic to itself. It need not look Euclidean in the affine chart: grid lines bend and spacing changes. What matters for the Cayley-Klein measurement is that the boundary intersections and the conic form are preserved.

The figure uses a Lorentz boost. The boundary conic is unchanged, while an interior grid and two sample points are moved by a projective transformation. The numeric check compares the reference-free `Psi` measurement before and after the motion.


In [ ]:
u = 0.62
motion_T = np.array([
    [math.cosh(u), 0.0, math.sinh(u)],
    [0.0, 1.0, 0.0],
    [math.sinh(u), 0.0, math.cosh(u)],
])
motion_conic_residual = float(np.max(np.abs(motion_T.T @ A @ motion_T - A)))
mp = hpoint(-0.25, 0.25)
mq = hpoint(0.35, -0.10)
psi_before = psi_value(mp, mq)
psi_after = psi_value(motion_T @ mp, motion_T @ mq)
checks["motion_conic_residual"] = motion_conic_residual
checks["motion_psi_residual"] = float(abs(psi_before - psi_after))

fig, ax = plt.subplots(figsize=(7.4, 5.6))
plot_unit_circle(ax, color=COLORS["blue"], linewidth=2.3, label="absolute before and after")
for c in np.linspace(-0.75, 0.75, 7):
    span = math.sqrt(max(0.0, 0.95**2 - c**2))
    ys = np.linspace(-span, span, 120)
    xs = np.full_like(ys, c)
    xy = np.column_stack([xs, ys])
    image = transform_points(motion_T, xy)
    ax.plot(xs, ys, color=COLORS["light"], linewidth=0.8)
    ax.plot(image[:, 0], image[:, 1], color=COLORS["green"], alpha=0.72, linewidth=1.0)
    xs = np.linspace(-span, span, 120)
    ys = np.full_like(xs, c)
    xy = np.column_stack([xs, ys])
    image = transform_points(motion_T, xy)
    ax.plot(xs, ys, color=COLORS["light"], linewidth=0.8)
    ax.plot(image[:, 0], image[:, 1], color=COLORS["green"], alpha=0.72, linewidth=1.0)
for label, point, color in [("p", mp, COLORS["ink"]), ("q", mq, COLORS["ink"]), ("T p", motion_T @ mp, COLORS["red"]), ("T q", motion_T @ mq, COLORS["red"])]:
    xy = aff(point)
    ax.scatter([xy[0]], [xy[1]], s=60, color=color, zorder=5)
    ax.text(xy[0] + 0.035, xy[1] + 0.035, label, fontsize=9, color=color)
ax.text(-1.16, -1.20, f"max |T.T A T - A| = {motion_conic_residual:.1e};  |Psi(p,q)-Psi(Tp,Tq)| = {abs(psi_before-psi_after):.1e}", fontsize=9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.25, 1.25)
ax.set_ylim(-1.27, 1.23)
ax.set_title("A projective motion may distort the affine grid while preserving the absolute")
ax.legend(loc="upper left")
motion_path = save_fig(fig, "figures", "absolute-preserving-motion.png")
display_artifact(motion_path, width=760)


In [ ]:
G = nx.DiGraph()
G.add_edges_from([
    ("T preserves A and B", "absolute intersections map as a set"),
    ("line pq maps to line T(p)T(q)", "absolute intersections map as a set"),
    ("absolute intersections map as a set", "cross-ratio class preserved"),
    ("projective maps preserve cross-ratio", "cross-ratio class preserved"),
    ("cross-ratio class preserved", "unoriented distance/angle preserved"),
    ("pole/polar pair", "reflection matrix"),
    ("reflection matrix", "T^2 is scalar identity"),
    ("reflection matrix", "T preserves A and B"),
    ("T preserves A and B", "Psi invariant"),
    ("two reflections", "K-motion product"),
    ("K-motion product", "rotation or translation behavior"),
])
positions = {
    "T preserves A and B": (0.0, 1.4),
    "line pq maps to line T(p)T(q)": (-2.2, 0.75),
    "absolute intersections map as a set": (-1.2, 0.15),
    "projective maps preserve cross-ratio": (-2.2, -0.55),
    "cross-ratio class preserved": (-0.35, -0.62),
    "unoriented distance/angle preserved": (0.9, -1.05),
    "pole/polar pair": (1.25, 0.75),
    "reflection matrix": (2.25, 0.2),
    "T^2 is scalar identity": (2.9, -0.55),
    "Psi invariant": (0.95, 0.35),
    "two reflections": (2.0, 1.25),
    "K-motion product": (3.05, 0.85),
    "rotation or translation behavior": (3.45, 0.1),
}
fig, ax = plt.subplots(figsize=(10.6, 5.8))
ax.set_axis_off()
nx.draw_networkx_edges(G, positions, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=13, edge_color=COLORS["gray"], width=1.4)
for i, node in enumerate(G.nodes):
    x, y = positions[node]
    color = [COLORS["blue"], COLORS["green"], COLORS["gold"], COLORS["purple"], COLORS["red"]][i % 5]
    ax.scatter([x], [y], s=1450, color=color, alpha=0.14, edgecolor=color, linewidth=1.2, zorder=2)
    ax.text(x, y, node, ha="center", va="center", fontsize=8.5, wrap=True, zorder=3)
ax.set_title("Proof dependency view: why preserving the absolute preserves measurement")
checks["proof_graph_nodes"] = G.number_of_nodes()
checks["proof_graph_edges"] = G.number_of_edges()
proof_graph_path = save_fig(fig, "figures", "motion-invariant-proof-dependencies.png")
display_artifact(proof_graph_path, width=880)


## 3. Eliminating the Reference Points `X` and `Y`

The source chapter next removes a practical nuisance. To compute `(p, q; X, Y)` one could intersect the line `pq` with the absolute every time, but the same reciprocal class is already encoded in the quadratic form `A`.

For `Omega_ij = i.T @ A @ j` and `Delta = Omega_pq^2 - Omega_pp Omega_qq`, the expression

`(Omega_pq + sqrt(Delta)) / (Omega_pq - sqrt(Delta))`

matches the endpoint cross-ratio up to the expected reciprocal choice. This is a useful translation: the metric comparison becomes a calculation in the point coordinates and the fundamental form, with no explicit construction of `X` and `Y`.


In [ ]:
sample_pairs = [
    ("A", hpoint(-0.65, -0.20), hpoint(0.10, 0.50)),
    ("B", hpoint(-0.30, 0.45), hpoint(0.62, 0.08)),
    ("C", hpoint(-0.55, 0.25), hpoint(0.35, -0.42)),
    ("D", hpoint(0.05, -0.58), hpoint(0.68, 0.18)),
]
rows = []
for name, a, b in sample_pairs:
    direct, pair_roots = chord_cross_ratio(a, b)
    formula = reference_free_alpha(a, b)
    sym_direct = direct + 1 / direct
    sym_formula = formula + 1 / formula
    rows.append({
        "pair": name,
        "direct_alpha": float(np.real(direct)),
        "formula_alpha": float(np.real(formula)),
        "reciprocal_class_error": reciprocal_class_error(direct, formula),
        "symmetric_value_error": float(abs(sym_direct - sym_formula)),
        "psi_value": psi_value(a, b),
    })
reference_table_path = save_table(rows, ARTIFACT_ROOT, "tables", "reference-free-measurement-samples.csv")
artifact_paths.append(reference_table_path)
checks["reference_formula_max_error"] = float(max(row["reciprocal_class_error"] for row in rows))

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.8), gridspec_kw={"width_ratios": [1.0, 1.15]})
ax = axes[0]
plot_unit_circle(ax, color=COLORS["blue"], linewidth=2.0)
for name, a, b in sample_pairs:
    direct, pair_roots = chord_cross_ratio(a, b)
    Xp = a + pair_roots[0] * (b - a)
    Yp = a + pair_roots[1] * (b - a)
    aa, bb = aff(a), aff(b)
    ax.plot([aff(Xp)[0], aff(Yp)[0]], [aff(Xp)[1], aff(Yp)[1]], color=COLORS["gray"], alpha=0.45, linewidth=1.0)
    ax.plot([aa[0], bb[0]], [aa[1], bb[1]], linewidth=2.0, label=f"pair {name}")
    mid = 0.5 * (aa + bb)
    ax.text(mid[0], mid[1], name, fontsize=9, weight="bold")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.18, 1.18)
ax.set_title("Endpoint construction")
ax = axes[1]
labels = [row["pair"] for row in rows]
x = np.arange(len(rows))
width = 0.35
direct_sym = [row["direct_alpha"] + 1 / row["direct_alpha"] for row in rows]
formula_sym = [row["formula_alpha"] + 1 / row["formula_alpha"] for row in rows]
ax.bar(x - width / 2, direct_sym, width=width, color=COLORS["blue"], alpha=0.75, label="from X,Y")
ax.bar(x + width / 2, formula_sym, width=width, color=COLORS["green"], alpha=0.75, label="from Omega,Delta")
ax.set_xticks(x, labels)
ax.set_ylabel("alpha + 1/alpha")
ax.set_title("Reference-free formula lands in the same reciprocal class")
ax.legend()
for row, xx in zip(rows, x):
    y = max(row["direct_alpha"] + 1 / row["direct_alpha"], row["formula_alpha"] + 1 / row["formula_alpha"]) + 0.08
    ax.text(xx, y, f"err {row['reciprocal_class_error']:.0e}", ha="center", fontsize=8)
fig.tight_layout()
reference_path = save_fig(fig, "figures", "reference-point-elimination.png")
display_artifact(reference_path, width=880)
display_artifact(reference_table_path)


## 4. Comparing Measurements With `Psi`

The expression using `Delta` works cleanly for nondegenerate forms, but the chapter also needs a formula that survives degenerations. Replacing the discriminant by the dual-conic expression gives

`Psi(p, q) = ((p x q).T @ B @ (p x q)) / ((p.T @ A @ p) * (q.T @ A @ q))`.

For the Euclidean embedding with `A = diag(0, 0, 1)` and `B = diag(1, 1, 0)`, `Psi` is exactly squared Euclidean distance. If `B = diag(1, -1, 0)`, it becomes the pseudo-Euclidean square `dx^2 - dy^2`. The level-set artifact below is not using the absolute conic picture for decoration; it is showing what the comparison formula says when distance is degenerate in the primal form but still meaningful through `B`.


In [ ]:
A_euclid = np.diag([0.0, 0.0, 1.0])
B_euclid = np.diag([1.0, 1.0, 0.0])
B_pseudo = np.diag([-1.0, 1.0, 0.0])
base = hpoint(-0.25, -0.10)
xs = np.linspace(-2.2, 2.2, 420)
ys = np.linspace(-2.0, 2.0, 380)
xx, yy = np.meshgrid(xs, ys)
euclid_grid = (xx - aff(base)[0]) ** 2 + (yy - aff(base)[1]) ** 2
pseudo_grid = (xx - aff(base)[0]) ** 2 - (yy - aff(base)[1]) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.8))
levels = [0.25, 1.0, 2.25, 4.0]
cs = axes[0].contour(xx, yy, euclid_grid, levels=levels, colors=[COLORS["green"]], linewidths=1.6)
axes[0].clabel(cs, inline=True, fontsize=8, fmt="%.2g")
axes[0].scatter([aff(base)[0]], [aff(base)[1]], color=COLORS["ink"], s=55)
axes[0].set_title("Euclidean degeneration: Psi = dx^2 + dy^2")
cs1 = axes[1].contour(xx, yy, pseudo_grid, levels=[-4, -2.25, -1, 0, 1, 2.25, 4], colors=[COLORS["red"]], linewidths=1.4)
axes[1].clabel(cs1, inline=True, fontsize=8, fmt="%.2g")
axes[1].scatter([aff(base)[0]], [aff(base)[1]], color=COLORS["ink"], s=55)
axes[1].set_title("Pseudo-Euclidean: Psi = dx^2 - dy^2")
for ax in axes:
    ax.axhline(aff(base)[1], color=COLORS["gray"], alpha=0.25)
    ax.axvline(aff(base)[0], color=COLORS["gray"], alpha=0.25)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-2.15, 2.15)
    ax.set_ylim(-1.85, 1.85)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.tight_layout()
level_path = save_fig(fig, "figures", "measurement-comparison-level-sets.png")

probe = hpoint(1.10, 0.55)
euclid_direct = float(np.sum((aff(probe) - aff(base)) ** 2))
pseudo_direct = float((aff(probe)[0] - aff(base)[0]) ** 2 - (aff(probe)[1] - aff(base)[1]) ** 2)
checks["euclidean_degenerate_psi_error"] = float(abs(psi_value(base, probe, A_euclid, B_euclid) - euclid_direct))
checks["pseudo_degenerate_psi_error"] = float(abs(psi_value(base, probe, A_euclid, B_pseudo) - pseudo_direct))
display_artifact(level_path, width=880)


## 5. Reflections From Pole/Polar Pairs

A projective involution has a fixed mirror line and a fixed center. For a Cayley-Klein reflection these two objects must form a pole/polar pair with respect to the fundamental object. In the nondegenerate model this is simply `m = A @ o` and `o = B @ m`, up to scale.

Given a center `o` and mirror `m`, the point matrix is

`T = <o,m> I - 2 o m.T`.

The artifact tracks one point and its image. The image lies on the same line through the center, and the mirror intersection is the harmonic partner required by the involution. The checks record the matrix identity, conic preservation, harmonic cross-ratio, and `Psi` preservation.


In [ ]:
center = hpoint(1.45, 0.35)
mirror = A @ center
reflection_T = reflection_matrix(center, mirror)
reflection_scale = float(mirror @ center)

rp = hpoint(0.10, 0.45)
rq = hpoint(-0.35, -0.20)
rp_image = reflection_T @ rp
rq_image = reflection_T @ rq
line_op = join(center, rp)
mirror_hit = meet(line_op, mirror)
local_basis = np.column_stack([rp, center])
coords = {
    "p": np.linalg.lstsq(local_basis, rp, rcond=None)[0],
    "p_image": np.linalg.lstsq(local_basis, rp_image, rcond=None)[0],
    "center": np.linalg.lstsq(local_basis, center, rcond=None)[0],
    "mirror_hit": np.linalg.lstsq(local_basis, mirror_hit, rcond=None)[0],
}
harmonic_value = cross_ratio_homogeneous_2d(coords["p"], coords["p_image"], coords["center"], coords["mirror_hit"])

checks["reflection_involution_residual"] = float(np.max(np.abs(reflection_T @ reflection_T - reflection_scale**2 * E3)))
checks["reflection_conic_residual"] = float(np.max(np.abs(reflection_T.T @ A @ reflection_T - reflection_scale**2 * A)))
checks["reflection_psi_residual"] = float(abs(psi_value(rp, rq) - psi_value(rp_image, rq_image)))
checks["reflection_harmonic_error"] = float(abs(harmonic_value + 1.0))
checks["pole_polar_residual"] = float(np.max(np.abs(mirror - A @ center)) + np.max(np.abs(center - B @ mirror)))

fig, ax = plt.subplots(figsize=(7.6, 5.8))
plot_unit_circle(ax, color=COLORS["blue"], linewidth=2.1, label="absolute conic")
plot_projective_line(ax, mirror, xlim=(-1.25, 1.75), ylim=(-1.2, 1.2), color=COLORS["green"], linewidth=2.0, label="mirror m = A o")
for point, image, color in [(rp, rp_image, COLORS["red"]), (rq, rq_image, COLORS["purple"])]:
    axy = aff(point)
    bxy = aff(image)
    ax.plot([axy[0], bxy[0]], [axy[1], bxy[1]], color=color, linewidth=1.6, alpha=0.7)
    ax.scatter([axy[0]], [axy[1]], color=color, s=54, zorder=5)
    ax.scatter([bxy[0]], [bxy[1]], facecolors="white", edgecolors=color, linewidth=1.8, s=62, zorder=5)
    plot_projective_line(ax, join(center, point), xlim=(-1.25, 1.75), ylim=(-1.2, 1.2), color=color, alpha=0.25, linestyle="--")
center_xy = aff(center)
hit_xy = aff(mirror_hit)
ax.scatter([center_xy[0]], [center_xy[1]], color=COLORS["ink"], s=68, zorder=5)
ax.text(center_xy[0] + 0.04, center_xy[1] + 0.02, "center o", fontsize=9, weight="bold")
ax.scatter([hit_xy[0]], [hit_xy[1]], color=COLORS["gold"], s=54, zorder=5)
ax.text(hit_xy[0] + 0.04, hit_xy[1] - 0.08, "mirror hit", fontsize=9, color=COLORS["gold"])
ax.text(-1.18, -1.18, f"T^2 residual {checks['reflection_involution_residual']:.1e}; harmonic residual {checks['reflection_harmonic_error']:.1e}", fontsize=9)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-1.25, 1.75)
ax.set_ylim(-1.25, 1.24)
ax.set_title("Pole/polar reflection: center, mirror, and harmonic image")
ax.legend(loc="upper left")
reflection_path = save_fig(fig, "figures", "pole-polar-reflection.png")
display_artifact(reflection_path, width=780)


In [ ]:
Asp = sp.diag(1, 1, -1)
osp = sp.Matrix([sp.Rational(3, 2), sp.Rational(1, 2), 1])
msp = Asp * osp
ssp = (msp.T * osp)[0]
Tsp = ssp * sp.eye(3) - 2 * osp * msp.T
symbolic_reflection_ok = (sp.simplify(Tsp * Tsp - ssp**2 * sp.eye(3)) == sp.zeros(3)) and (sp.simplify(Tsp.T * Asp * Tsp - ssp**2 * Asp) == sp.zeros(3))

m0 = sp.Matrix([0, 1, 0])
m30 = sp.Matrix([-sp.Rational(1, 2), sp.sqrt(3) / 2, 0])
def sym_reflection_from_mirror(m):
    o = Asp.inv() * m
    s = (m.T * o)[0]
    return sp.simplify(s * sp.eye(3) - 2 * o * m.T)
R_exact = sp.simplify(sym_reflection_from_mirror(m30) * sym_reflection_from_mirror(m0))
R_expected = sp.Matrix([[sp.Rational(1, 2), -sp.sqrt(3) / 2, 0], [sp.sqrt(3) / 2, sp.Rational(1, 2), 0], [0, 0, 1]])
symbolic_rotation_ok = sp.simplify(R_exact - R_expected) == sp.zeros(3)
checks["symbolic_reflection_identity"] = bool(symbolic_reflection_ok)
checks["symbolic_two_reflection_rotation"] = bool(symbolic_rotation_ok)
{"symbolic_reflection_identity": symbolic_reflection_ok, "symbolic_two_reflection_rotation": symbolic_rotation_ok}


## 6. Applied Lab: Two Reflections Produce Rotation Behavior

The last section of the chapter studies what happens when reflections are composed. The Euclidean rule is familiar: two line reflections through intersecting mirrors produce a rotation by twice the mirror angle, while parallel mirrors produce a translation. The same projective mechanism persists in Cayley-Klein geometries, but the location of the mirror intersection relative to the absolute changes the qualitative behavior.

The HTML lab compares two cases in the disk model. On the left, the mirrors meet at the disk center, so the product is an ordinary finite-center rotation. On the right, the mirrors meet outside the absolute; iterating their product pushes sample points toward boundary fixed directions, a hyperbolic-type analogue of the mirror-chain behavior.


In [ ]:
def iterates(matrix, point, count):
    pts = []
    current = np.asarray(point, dtype=float)
    for _ in range(count):
        pts.append(aff(current))
        current = matrix @ current
    return np.array(pts)

phi1 = 0.0
phi2 = math.pi / 6.0
m_left_1 = mirror_line_through_angle(phi1)
m_left_2 = mirror_line_through_angle(phi2)
S1, _ = reflection_from_mirror(m_left_1)
S2, _ = reflection_from_mirror(m_left_2)
R = S2 @ S1
rotation_angle = math.atan2(R[1, 0], R[0, 0])
rotation_angle_error = abs(rotation_angle - 2 * (phi2 - phi1))
left_orbit = iterates(R, hpoint(0.48, 0.15), 7)

m_right_1 = np.array([-0.7, 1.0, 0.91])
m_right_2 = np.array([0.7, 1.0, -0.91])
H1, _ = reflection_from_mirror(m_right_1)
H2, _ = reflection_from_mirror(m_right_2)
H = H2 @ H1
right_orbit = iterates(H, hpoint(0.00, 0.25), 10)
checks["two_reflection_rotation_angle_error"] = float(rotation_angle_error)
checks["two_reflection_motion_residual"] = float(np.max(np.abs(R.T @ A @ R - A)))
right_scale = (m_right_1 @ B @ m_right_1) * (m_right_2 @ B @ m_right_2)
checks["hyperbolic_product_preserves_disk_form_residual"] = float(np.max(np.abs((H.T @ A @ H) / (right_scale**2) - A)))

orbit_rows = []
for mode, orbit in [("finite_center_rotation", left_orbit), ("exterior_center_iteration", right_orbit)]:
    for index, xy in enumerate(orbit):
        orbit_rows.append({"mode": mode, "step": index, "x": float(xy[0]), "y": float(xy[1]), "norm": float(np.linalg.norm(xy))})
orbit_table_path = save_table(orbit_rows, ARTIFACT_ROOT, "tables", "reflection-orbit-summary.csv")
artifact_paths.append(orbit_table_path)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Finite mirror intersection: rotation", "Exterior intersection: hyperbolic-type product"))
theta = np.linspace(0, 2 * np.pi, 361)
for col in [1, 2]:
    fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines", line=dict(color=COLORS["blue"], width=3), name="absolute" if col == 1 else "absolute ", showlegend=(col == 1)), row=1, col=col)
for line, name, col in [(m_left_1, "mirror 1", 1), (m_left_2, "mirror 2", 1), (m_right_1, "mirror 1 ", 2), (m_right_2, "mirror 2 ", 2)]:
    seg = line_disk_segment(line)
    if len(seg):
        fig.add_trace(go.Scatter(x=seg[:, 0], y=seg[:, 1], mode="lines", line=dict(width=3), name=name, showlegend=(col == 1)), row=1, col=col)
fig.add_trace(go.Scatter(x=left_orbit[:, 0], y=left_orbit[:, 1], mode="lines+markers+text", text=[str(i) for i in range(len(left_orbit))], textposition="top center", marker=dict(size=8, color=np.arange(len(left_orbit)), colorscale="Viridis"), line=dict(color=COLORS["green"], width=2), name="rotation orbit"), row=1, col=1)
fig.add_trace(go.Scatter(x=right_orbit[:, 0], y=right_orbit[:, 1], mode="lines+markers+text", text=[str(i) for i in range(len(right_orbit))], textposition="top center", marker=dict(size=8, color=np.arange(len(right_orbit)), colorscale="Plasma"), line=dict(color=COLORS["red"], width=2), name="exterior-center orbit"), row=1, col=2)
fig.update_xaxes(range=[-1.08, 1.08], constrain="domain")
fig.update_yaxes(range=[-1.08, 1.08], scaleanchor="x", scaleratio=1)
fig.update_layout(height=520, width=980, title_text="Two Cayley-Klein reflections compose to a motion", template="plotly_white", legend=dict(orientation="h", y=-0.12))
html_path = artifact_path(ARTIFACT_ROOT, "html", "two-reflections-rotation-lab.html")
fig.write_html(html_path, include_plotlyjs=True, full_html=True)
artifact_paths.append(html_path)

display_artifact(html_path, width="100%", height=560)
display_artifact(orbit_table_path)


## Artifact Gallery

These direct links keep the notebook readable before execution and make the artifact set easy to audit.

![Oriented reciprocal measurement](../../artifacts/chapter-21-measurements-and-transformations/figures/oriented-measurement-reciprocal.png)

![Absolute-preserving motion](../../artifacts/chapter-21-measurements-and-transformations/figures/absolute-preserving-motion.png)

![Reference-point elimination](../../artifacts/chapter-21-measurements-and-transformations/figures/reference-point-elimination.png)

![Measurement comparison level sets](../../artifacts/chapter-21-measurements-and-transformations/figures/measurement-comparison-level-sets.png)

![Pole/polar reflection](../../artifacts/chapter-21-measurements-and-transformations/figures/pole-polar-reflection.png)

Open the [two-reflections rotation lab](../../artifacts/chapter-21-measurements-and-transformations/html/two-reflections-rotation-lab.html) and the generated tables when comparing finite-center and exterior-center behavior.


## Takeaways

A Cayley-Klein measurement is projective data plus a chosen absolute. Orientation choices matter only until the reciprocal class is recognized. Once measurements are expressed through `A`, `B`, and homogeneous coordinates, the preservation problem becomes concrete matrix algebra.

Reflections are the most useful test case. The pole/polar condition gives a reflection matrix, the matrix is an involution up to projective scale, and the same matrix preserves `Psi`. Products of two such reflections are motions again. Their visible behavior depends on where the mirror intersection sits relative to the absolute: inside, on, or outside the conic.


In [ ]:
required_numeric_checks = {
    "orientation_reciprocal_error": 1e-9,
    "motion_conic_residual": 1e-9,
    "motion_psi_residual": 1e-9,
    "reference_formula_max_error": 1e-9,
    "euclidean_degenerate_psi_error": 1e-9,
    "pseudo_degenerate_psi_error": 1e-9,
    "reflection_involution_residual": 1e-9,
    "reflection_conic_residual": 1e-9,
    "reflection_psi_residual": 1e-9,
    "reflection_harmonic_error": 1e-9,
    "pole_polar_residual": 1e-9,
    "two_reflection_rotation_angle_error": 1e-9,
    "two_reflection_motion_residual": 1e-9,
    "hyperbolic_product_preserves_disk_form_residual": 1e-8,
}
for key, tolerance in required_numeric_checks.items():
    assert checks[key] < tolerance, (key, checks[key], tolerance)
assert checks["symbolic_reflection_identity"]
assert checks["symbolic_two_reflection_rotation"]
assert checks["proof_graph_nodes"] >= 10 and checks["proof_graph_edges"] >= 10

assert_artifacts(artifact_paths, min_size=256)
raster_paths = [path for path in artifact_paths if path.suffix.lower() == ".png"]
raster_stats = []
for path in raster_paths:
    stats = image_stats(path)
    assert stats["width"] >= 200 and stats["height"] >= 150
    assert stats["pixel_std"] > 1.0
    stats["path"] = rel(path)
    raster_stats.append(stats)

visual_checks = {
    "chapter": 21,
    "source_span": "printed pp. 399-422 / PDF pp. 421-444",
    "all_files_exist": all(path.exists() for path in artifact_paths),
    "artifact_count": len(artifact_paths),
    "raster_artifacts": raster_stats,
    "html_artifact": rel(html_path),
    "table_artifacts": [rel(reference_table_path), rel(orbit_table_path)],
    "cross_ratio_error": max(checks["orientation_reciprocal_error"], checks["reference_formula_max_error"]),
    "numeric_checks": checks,
    "libraries": ["numpy", "matplotlib", "sympy", "networkx", "plotly"],
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")
final = {
    "chapter": 21,
    "notebook_executed": True,
    "source_span": "Chapter 21, Sections 21.1-21.6; printed pp. 399-422; PDF pp. 421-444",
    "storyboard": rel(storyboard_path),
    "visual_checks": rel(visual_checks_path),
    "artifacts": [rel(path) for path in artifact_paths],
    "checks": checks,
}
final_path = save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final
